# 21. Descriptors and slots

**Level:** Advanced

**Learning outcome:** Reuse attribute validation with descriptors and understand what slots restrict and do not guarantee.

**Prerequisites:** 03, 05, and 12: properties, inheritance, and object protocols.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](20_Serialization_Schemas_Enums_and_Snapshots.ipynb) | [Next notebook](22_Cached_Properties_and_Invalidation.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Advanced

**Learning goal:** Understand descriptors, slots, caching, __new__, and class-creation hooks without overusing them.

**Descriptors** define attribute behavior through methods such as __get__, __set__, and
__set_name__. Properties and ordinary bound methods rely on descriptor mechanisms. A
reusable validating descriptor can enforce the same invariant across several fields. Access
through the class should normally return the descriptor itself when instance is None.

**Attribute lookup** is richer than 'instance dictionary then class'. Data descriptors can
take precedence over instance dictionary entries; non-data descriptors such as ordinary
methods participate differently. __getattr__ is a fallback for missing attributes.
__getattribute__ intercepts ordinary attribute access and is easy to make recursively broken.
Prefer a property or explicit method unless interception is truly required.

**__slots__** declares a set of attribute storage names for instances and can remove their
ordinary __dict__ when the hierarchy permits it. It is not a security boundary, deep
immutability, or a universal performance fix. A base class with a dictionary can change the
result. Weak references and inheritance require deliberate design.

**Cached properties** compute once and store a result. They suit expensive derived values only
when invalidation semantics are clear. A cached value can become stale if inputs change.
Concurrent access may also compute a value more than once, so do not rely on exactly-once
side effects.

**Creation hooks.** __new__ creates instances and is useful when subclassing immutable built-ins.
__init_subclass__ customizes subclass creation without requiring a custom metaclass.
A metaclass controls class creation itself; `type(SomeClass)` is its metaclass. Most network
automation should prefer ordinary classes and explicit registries over metaclass machinery.

**Pause and explain:** Choose a property before a descriptor unless reuse justifies it; explain cache invalidation.

## Worked-example structure

This notebook contains **2 worked patterns and 20 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 076: Reusable validating descriptor

**Core concept:** One descriptor can validate assignment to a declared model field.

**Network-engineering purpose:** Several numeric configuration fields can share range-checking machinery.

### Read the implementation in this order
1. Capture a backing name with __set_name__.
2. Return self for access through the class.
3. Validate before writing through instance storage.
4. Check valid construction and preserved state after rejection.

**Common mistake / boundary:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

**Practice after the ten worked scenarios:** Add a second descriptor field with a different range and test both independently.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0751 - Reusable validating descriptor: branch scenario

**Scenario 01/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [1]:
# Example 0751: Reusable validating descriptor - branch
case = {'mtu': 1500}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


1500


### Example 0752 - Reusable validating descriptor: campus scenario

**Scenario 02/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
9000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [2]:
# Example 0752: Reusable validating descriptor - campus
case = {'mtu': 9000}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


9000


### Example 0753 - Reusable validating descriptor: dc scenario

**Scenario 03/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
9216
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [3]:
# Example 0753: Reusable validating descriptor - dc
case = {'mtu': 9216}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


9216


### Example 0754 - Reusable validating descriptor: wan scenario

**Scenario 04/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
576
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [4]:
# Example 0754: Reusable validating descriptor - wan
case = {'mtu': 576}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


576


### Example 0755 - Reusable validating descriptor: lab scenario

**Scenario 05/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
1600
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [5]:
# Example 0755: Reusable validating descriptor - lab
case = {'mtu': 1600}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


1600


### Example 0756 - Reusable validating descriptor: wireless scenario

**Scenario 06/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [6]:
# Example 0756: Reusable validating descriptor - wireless
case = {'mtu': 1500}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


1500


### Example 0757 - Reusable validating descriptor: remote scenario

**Scenario 07/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
2000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [7]:
# Example 0757: Reusable validating descriptor - remote
case = {'mtu': 2000}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


2000


### Example 0758 - Reusable validating descriptor: core scenario

**Scenario 08/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
4470
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [8]:
# Example 0758: Reusable validating descriptor - core
case = {'mtu': 4470}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


4470


### Example 0759 - Reusable validating descriptor: mgmt scenario

**Scenario 09/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
9100
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [9]:
# Example 0759: Reusable validating descriptor - mgmt
case = {'mtu': 9100}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


9100


### Example 0760 - Reusable validating descriptor: backup scenario

**Scenario 10/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
8192
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [10]:
# Example 0760: Reusable validating descriptor - backup
case = {'mtu': 8192}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


8192


## Pattern 077: Slots catch accidental field names

**Core concept:** A dictionary-free slotted object restricts ordinary instance attribute names.

**Network-engineering purpose:** A misspelled hostname field can fail immediately instead of silently creating a new attribute.

### Read the implementation in this order
1. Declare slots on a direct object subclass.
2. Initialize the declared field.
3. Try writing a misspelled field.
4. Verify AttributeError and the original field.

**Common mistake / boundary:** Slots do not make values immutable and do not replace validation or access control.

**Practice after the ten worked scenarios:** Compare vars() behavior for a normal instance and a slotted instance.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0761 - Slots catch accidental field names: branch scenario

**Scenario 01/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [11]:
# Example 0761: Slots catch accidental field names - branch
case = {'hostname': 'branch-edge-01'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0762 - Slots catch accidental field names: campus scenario

**Scenario 02/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [12]:
# Example 0762: Slots catch accidental field names - campus
case = {'hostname': 'campus-access-02'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0763 - Slots catch accidental field names: dc scenario

**Scenario 03/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [13]:
# Example 0763: Slots catch accidental field names - dc
case = {'hostname': 'dc-leaf-03'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0764 - Slots catch accidental field names: wan scenario

**Scenario 04/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [14]:
# Example 0764: Slots catch accidental field names - wan
case = {'hostname': 'wan-border-04'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0765 - Slots catch accidental field names: lab scenario

**Scenario 05/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [15]:
# Example 0765: Slots catch accidental field names - lab
case = {'hostname': 'lab-router-05'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0766 - Slots catch accidental field names: wireless scenario

**Scenario 06/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [16]:
# Example 0766: Slots catch accidental field names - wireless
case = {'hostname': 'wireless-dist-06'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0767 - Slots catch accidental field names: remote scenario

**Scenario 07/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [17]:
# Example 0767: Slots catch accidental field names - remote
case = {'hostname': 'remote-gateway-07'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0768 - Slots catch accidental field names: core scenario

**Scenario 08/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [18]:
# Example 0768: Slots catch accidental field names - core
case = {'hostname': 'core-router-08'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0769 - Slots catch accidental field names: mgmt scenario

**Scenario 09/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [19]:
# Example 0769: Slots catch accidental field names - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0770 - Slots catch accidental field names: backup scenario

**Scenario 10/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [20]:
# Example 0770: Slots catch accidental field names - backup
case = {'hostname': 'backup-edge-10'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


## Check your understanding: Descriptors and slots

**You should now be able to:** Reuse attribute validation with descriptors and understand what slots restrict and do not guarantee.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.